# Chapter 05 · Mathematics: Vectors, matrices, and model functions

[Chapter guide](README.md) · [Problem-to-mathematics guide](../MATHEMATICS.md) · [Course index](../README.md)

**Status: optional reference for later; not yet reviewed.** The programming lessons are already covered. Continue the programming-led mentorship without needing to complete this companion first. Use small numerical explanations when helpful, and revisit formal notation gradually.

Read one section at a time: words → a small example → symbols → code → your explanation. Symbols are shorthand for ideas you already used. You do not need to memorize all the formulas before continuing.

**Prerequisites:** Weighted sums and gradients; new vector and matrix notation is explained here.

The runnable examples use only Python and the installed PyTorch. Start in a fresh kernel and run Setup, then proceed in order. Each example keeps its calculations small and supplies its own data. Code cells are saved unexecuted so you can make your own predictions first.

## Contents

1. [1. Two features become a weighted sum](#weighted-sum)
2. [2. Read batch and layer shapes](#matrix-shapes)
3. [3. A feature scales its weight gradient](#feature-gradients)
4. [4. Objects store a function and its state](#model-state)

## Setup

In [ ]:
import torch

<a id="weighted-sum"></a>
## 1. Two features become a weighted sum


An example with two features is an ordered pair $\mathbf x=(x_1,x_2)$; bold $\mathbf x$ means a vector. Its two weights are $\mathbf w=(w_1,w_2)$:

$$\hat y=w_1x_1+w_2x_2+b$$

A **dot product** means multiply corresponding entries, then add. Thus $\mathbf w\cdot\mathbf x$ is shorthand for $w_1x_1+w_2x_2$.

For input `(2, 4)`, weights `(3, 1)`, and bias `2`, prediction is $3(2)+1(4)+2=12$. The same weights and bias are used for every example.

The familiar rule can also be recovered from these exact toy observations: input `(0, 0)` with target `2` fixes $b=2$; input `(1, 0)` with target `5` then fixes $w_1=3$; input `(0, 1)` with target `3` fixes $w_2=1$. These are three simple equations about three unknown parameter values. Noisy or insufficient observations can change what is recoverable.

**Connection:** layer notebook 01 and two-feature notebook 03, Lessons 01–03.

**Before running:** calculate the prediction for `(3, 2)` using the same weights and bias.


**My prediction or explanation:** _write here_

In [ ]:
# Multiply matching features and weights, then add the shared bias.
input_vector = torch.tensor([3., 2.])
weight_vector = torch.tensor([3., 1.])
bias_value = 2.0
weighted_parts = weight_vector * input_vector
prediction = weighted_parts.sum() + bias_value
print("Weighted feature contributions:", weighted_parts.tolist())
print("Prediction:", prediction.item())

<a id="matrix-shapes"></a>
## 2. Read batch and layer shapes


A **matrix** is a table with rows and columns. For a batch, put one example on each row. Let $X$ have $N$ examples and $d$ input features, giving shape `[N, d]`.

A layer with $k$ outputs stores its weights as a matrix $W$ of shape `[k, d]`. Each row contains the weights for one output. The bias has shape `[k]`: one value per output. All examples share these values.

$$\hat Y=XW^T+b$$

$W^T$ means **transpose**: swap rows and columns. It has shape `[d, k]`, so multiplying `[N, d]` by `[d, k]` produces `[N, k]`. Add the same bias to each example row. You can always expand this shorthand back into weighted sums.

With bias enabled, the layer's scalar parameter count is:

$$\text{parameter count}=kd+k$$

`Linear(2, 1)` has `2 × 1 + 1 = 3` values; `Linear(2, 3)` has `2 × 3 + 3 = 9`. Increasing the batch size does not create additional weights.

**Connection:** layer notebook 01, training notebook 02, and two-feature notebook 03. See [Linear](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html).

**Before running:** predict the output shape for four examples, two features each, and one output per example.


**My prediction or explanation:** _write here_

In [ ]:
# Compare matrix notation with the output of a real Linear layer.
x_batch = torch.tensor([[0., 0.], [1., 0.], [0., 1.], [2., 4.]])
layer = torch.nn.Linear(2, 1)
with torch.no_grad():
    layer.weight.copy_(torch.tensor([[3., 1.]]))
    layer.bias.fill_(2.)
    matrix_predictions = x_batch @ layer.weight.T + layer.bias
    layer_predictions = layer(x_batch)
print("Input shape:", x_batch.shape)
print("Weight shape:", layer.weight.shape)
print("Transposed weight shape:", layer.weight.T.shape)
print("Output shape:", layer_predictions.shape)
print("Predictions:", layer_predictions.flatten().tolist())
print("Calculations agree:", torch.allclose(matrix_predictions, layer_predictions))

<a id="feature-gradients"></a>
## 3. A feature scales its weight gradient


For one example with error $e$, each weight gradient depends on its own feature:

$$g_{w_1}=2ex_1,\qquad g_{w_2}=2ex_2,\qquad g_b=2e$$

If $x_2=0$, the second weight's contribution to this example's prediction is zero, and its gradient contribution is zero. That is not permanent: another example with nonzero $x_2$ may produce a nonzero contribution.

At weights `(1, 1)` and bias `0`, input `(1, 0)` targeting `5` has error `-4` and gradients `(-8, 0, -8)`. Input `(0, 1)` targeting `3` has error `-2` and gradients `(0, -4, -4)`. Averaging over this two-example batch gives `(-4, -2, -6)` for `(weight1, weight2, bias)`.

**Connection:** two-feature notebook 03, Lessons 04–08, and custom-model notebook 05, Lesson 04.

**Before running:** why is the averaged bias gradient `-6`, rather than the sum `-12`?


**My prediction or explanation:** _write here_

In [ ]:
# Compare expanded gradient formulas with the layer's stored gradients.
x_batch = torch.tensor([[1., 0.], [0., 1.]])
y_batch = torch.tensor([[5.], [3.]])
layer = torch.nn.Linear(2, 1)
with torch.no_grad():
    layer.weight.fill_(1.)
    layer.bias.fill_(0.)
layer.zero_grad(set_to_none=True)
errors = layer(x_batch) - y_batch
formula_weight_gradients = (2 * errors.detach() * x_batch).mean(dim=0)
formula_bias_gradient = (2 * errors.detach()).mean()
loss = (errors ** 2).mean()
loss.backward()
print("Formula weight gradients:", formula_weight_gradients.tolist())
print("Layer weight gradients:", layer.weight.grad.flatten().tolist())
print("Formula bias gradient:", formula_bias_gradient.item())
print("Layer bias gradient:", layer.bias.grad.item())

<a id="model-state"></a>
## 4. Objects store a function and its state


Mathematically, a model is a prediction function $f_\theta(x)$ plus a choice of parameter values $\theta$. In code, a class describes the structure and calculation, while each instance stores its own values. `forward` implements the calculation; `parameters()` exposes the registered values for training.

For our small linear model, its saved model state contains the weights and bias. A compatible class supplies the same formula so loading those values reproduces predictions. Other model types can also have state values that are not trainable parameters.

No-grad changes whether PyTorch records a calculation for backward. It does not change this layer's mathematical prediction or itself freeze parameter values. Evaluation mode is a separate choice about layer behavior; for a simple `Linear` layer it leaves the formula the same. See [autograd modes](https://docs.pytorch.org/docs/stable/notes/autograd.html#locally-disabling-gradient-computation).

A momentum optimizer also has memory beyond $\theta$, as Chapter 04's math explains. Reproducing the next training update can therefore require more than reproducing a prediction.

**Connection:** Python-class notebook 04, custom-model notebook 05, and no-grad notebook 06.

**Explain:** why can two separate model objects make the same predictions after loading the same state?


**My prediction or explanation:** _write here_

**My observation:** _write here_

Read every matrix expression as a collection of weighted sums. Shapes describe how examples, features, and outputs are arranged; they do not by themselves tell you the numerical parameter values.